In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
cd drive/MyDrive/Datalab/practice

/content/drive/MyDrive/Datalab/practice


In [ ]:

import os
import shutil
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim


In [ ]:
df_with_meta = pd.read_csv("../data/merged_sdg_finance_data.csv")

In [ ]:
df_with_meta.head(4)

,Country Name,Country Code,Series Code,1990,1991,1992,1993,1994,1995,1996,...,2013,2014,2015,2016,2017,2018,2019,2020,Region,Income Group
0,Afghanistan,AFG,DT.TDS.DPPF.XP.ZS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1.554784,1.875261,2.805854,3.735882,3.548648,2.595223,2.390422,2.244082,South Asia,Low income
1,Afghanistan,AFG,EG.ELC.ACCS.ZS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,68.290649,89.500000,71.500000,97.699997,97.699997,96.616135,97.699997,97.699997,South Asia,Low income
2,Afghanistan,AFG,GFDD.AI.01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,155.781150,172.807290,181.087280,182.145250,166.167050,174.299110,183.287310,NaN,South Asia,Low income
3,Afghanistan,AFG,GFDD.DI.04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,59.721950,59.328740,60.263810,65.520030,73.749060,84.970050,97.712900,96.255890,South Asia,Low income


### Scaling

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

# 1. 연도 컬럼 설정
year_cols = [str(y) for y in range(1990, 2021)]

# 2. 스케일러와 스케일링된 데이터프레임 초기화
scalers = {}
df_scaled = df_with_meta.copy()

# 3. 지표별(Series Code)로 스케일링 진행 (NaN 유지)
for series in df_scaled['Series Code'].unique():
    scaler = MinMaxScaler()
    idx = df_scaled[df_scaled['Series Code'] == series].index
    data = df_scaled.loc[idx, year_cols].values

    # 해당 지표에 데이터가 하나라도 있는 경우만 진행
    if not np.all(pd.isna(data)):
        # NaN을 포함한 상태로 학습 및 변환 (Sklearn은 이를 지원함)
        scaled_values = scaler.fit_transform(data.reshape(-1, 1)).reshape(data.shape)
        df_scaled.loc[idx, year_cols] = scaled_values
        scalers[series] = scaler

print(f"✅ 1단계 완료: NaN이 살아있는 df_scaled 생성됨.")
print(f"현재 결측치 수: {df_scaled[year_cols].isnull().sum().sum()}개") # 0보다 커야 함!

✅ 1단계 완료: NaN이 살아있는 df_scaled 생성됨.
현재 결측치 수: 31092개


### Linear imputer

In [ ]:
# 선형 보간 버전: 양옆의 숫자를 선으로 이어 결측치를 채움
df_linear_final = df_scaled.copy()
df_linear_final[year_cols] = df_linear_final[year_cols].interpolate(axis=1, limit_direction='both')

# 만약 끝까지 안 채워진 곳이 있다면 0으로 마무리
df_linear_final[year_cols] = df_linear_final[year_cols].fillna(0)

# 역스케일링 및 저장
for series, scaler in scalers.items():
    idx = df_linear_final[df_linear_final['Series Code'] == series].index
    vals = df_linear_final.loc[idx, year_cols].values
    df_linear_final.loc[idx, year_cols] = scaler.inverse_transform(vals.reshape(-1, 1)).reshape(vals.shape)

df_linear_final.to_csv("../data/WorldBank_Linear_Imputed.csv", index=False)
print("✅ 버전 1: 선형 보간 데이터 저장 완료")

✅ 버전 1: 선형 보간 데이터 저장 완료


### Tensor

In [ ]:
countries = df_scaled['Country Name'].unique()
series_list = df_scaled['Series Code'].unique()

# NaN으로 가득 찬 박스 생성
data_3d = np.full((len(countries), len(year_cols), len(series_list)), np.nan)

for i, country in enumerate(countries):
    for j, s_code in enumerate(series_list):
        row = df_scaled[(df_scaled['Country Name'] == country) & (df_scaled['Series Code'] == s_code)]
        if not row.empty:
            data_3d[i, :, j] = row[year_cols].values

print(f"✅ 텐서 준비 완료. NaN 개수: {np.isnan(data_3d).sum()}개")

✅ 텐서 준비 완료. NaN 개수: 31092개


In [ ]:
# 1. 라이브러리 설치 (맨 앞에 느낌표 포함)
!pip install pypots

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 749.8/749.8 kB 24.5 MB/s eta 0:00:00


In [ ]:
# 1. SAITS 실행 전 확인
print(f"SAITS 실행 전 결측치 개수: {np.isnan(data_3d).sum()}")

# 2. CSDI 실행 전 확인
# (만약 여기서 0이 나온다면, 이미 SAITS가 data_3d를 오염시킨 것입니다.)
print(f"CSDI 실행 전 결측치 개수: {np.isnan(data_3d).sum()}")

SAITS 실행 전 결측치 개수: 0
CSDI 실행 전 결측치 개수: 0


In [ ]:
from pypots.imputation import CSDI
# 1. 장치 설정 (이게 빠져서 에러가 난 거예요!)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# 2. 데이터 형태 확인 (data_3d가 준비되어 있어야 함)
num_features = data_3d.shape[2]
num_steps = data_3d.shape[1]

# 1. 모델 정의 (상세 파라미터 포함)
csdi = CSDI(
    n_features=data_3d.shape[2],
    n_steps=data_3d.shape[1],
    d_time_embedding=64,
    d_feature_embedding=64,
    d_diffusion_embedding=128,
    n_layers=2,
    n_heads=4,
    n_channels=64,
    epochs=50,
    patience=10,
    batch_size=32,
    device=device
)

# 2. 모델 학습 (fit)
print("🚀 CSDI 학습 시작...")
csdi.fit({"X": data_3d})

# 3. 보간 데이터 생성 (impute)
print("🪄 CSDI 보간 데이터 생성 중...")
imputed_csdi = csdi.impute({"X": data_3d})

# 딕셔너리 형태일 경우 추출
if isinstance(imputed_csdi, dict):
    imputed_csdi = imputed_csdi["imputation"]

2026-01-12 13:08:56 [INFO]: Using the given device: cuda
2026-01-12 13:08:56 [WARNING]: ‼️ saving_path not given. Model files and tensorboard file will not be saved.
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(
2026-01-12 13:08:56 [INFO]: CSDI initialized with the given hyperparameters, the number of trainable parameters: 222,145


🚀 CSDI 학습 시작...


2026-01-12 13:08:57 [INFO]: Epoch 001 - training loss (default): 0.8932
2026-01-12 13:08:57 [INFO]: Epoch 002 - training loss (default): 0.4616
2026-01-12 13:08:57 [INFO]: Epoch 003 - training loss (default): 0.2535
2026-01-12 13:08:58 [INFO]: Epoch 004 - training loss (default): 0.2602
2026-01-12 13:08:58 [INFO]: Epoch 005 - training loss (default): 0.2189
2026-01-12 13:08:58 [INFO]: Epoch 006 - training loss (default): 0.1791
2026-01-12 13:08:59 [INFO]: Epoch 007 - training loss (default): 0.1779
2026-01-12 13:08:59 [INFO]: Epoch 008 - training loss (default): 0.1197
2026-01-12 13:08:59 [INFO]: Epoch 009 - training loss (default): 0.1226
2026-01-12 13:08:59 [INFO]: Epoch 010 - training loss (default): 0.1313
2026-01-12 13:09:00 [INFO]: Epoch 011 - training loss (default): 0.1209
2026-01-12 13:09:00 [INFO]: Epoch 012 - training loss (default): 0.1765
2026-01-12 13:09:00 [INFO]: Epoch 013 - training loss (default): 0.1488
2026-01-12 13:09:00 [INFO]: Epoch 014 - training loss (default):

🪄 CSDI 보간 데이터 생성 중...


To install a Python library, you can use `pip install` followed by the library name. If you are in a Colab notebook, you can prefix the command with `!` to run it as a shell command.

In [ ]:
from pypots.imputation import SAITS

# 1. 모델 정의 (상세 파라미터 포함)
saits = SAITS(
    n_features=data_3d.shape[2],
    n_steps=data_3d.shape[1],
    n_layers=2,
    d_model=128,
    d_ffn=128,
    n_heads=4,
    d_k=32,
    d_v=32,
    dropout=0.1,
    epochs=50,
    patience=10,
    batch_size=32,
    device=device
)

# 2. 모델 학습 (fit)
print("🚀 SAITS 학습 시작...")
saits.fit({"X": data_3d})

# 3. 보간 데이터 생성 (impute)
print("🪄 SAITS 보간 데이터 생성 중...")
imputed_saits = saits.impute({"X": data_3d})

# 딕셔너리 형태일 경우 추출
if isinstance(imputed_saits, dict):
    imputed_saits = imputed_saits["imputation"]

2026-01-12 13:10:02 [INFO]: Using the given device: cuda
2026-01-12 13:10:02 [WARNING]: ‼️ saving_path not given. Model files and tensorboard file will not be saved.
2026-01-12 13:10:02 [INFO]: Using customized MAE as the training loss function.
2026-01-12 13:10:02 [INFO]: Using customized MSE as the validation metric function.
2026-01-12 13:10:02 [INFO]: SAITS initialized with the given hyperparameters, the number of trainable parameters: 406,468
2026-01-12 13:10:03 [INFO]: Epoch 001 - training loss (MAE): 0.6013


🚀 SAITS 학습 시작...


2026-01-12 13:10:03 [INFO]: Epoch 002 - training loss (MAE): 0.4101
2026-01-12 13:10:03 [INFO]: Epoch 003 - training loss (MAE): 0.3646
2026-01-12 13:10:03 [INFO]: Epoch 004 - training loss (MAE): 0.3407
2026-01-12 13:10:03 [INFO]: Epoch 005 - training loss (MAE): 0.3214
2026-01-12 13:10:03 [INFO]: Epoch 006 - training loss (MAE): 0.3009
2026-01-12 13:10:03 [INFO]: Epoch 007 - training loss (MAE): 0.2841
2026-01-12 13:10:03 [INFO]: Epoch 008 - training loss (MAE): 0.2674
2026-01-12 13:10:04 [INFO]: Epoch 009 - training loss (MAE): 0.2529
2026-01-12 13:10:04 [INFO]: Epoch 010 - training loss (MAE): 0.2391
2026-01-12 13:10:04 [INFO]: Epoch 011 - training loss (MAE): 0.2302
2026-01-12 13:10:04 [INFO]: Epoch 012 - training loss (MAE): 0.2208
2026-01-12 13:10:04 [INFO]: Epoch 013 - training loss (MAE): 0.2082
2026-01-12 13:10:04 [INFO]: Epoch 014 - training loss (MAE): 0.1954
2026-01-12 13:10:04 [INFO]: Epoch 015 - training loss (MAE): 0.1853
2026-01-12 13:10:04 [INFO]: Epoch 016 - training

🪄 SAITS 보간 데이터 생성 중...


In [ ]:
np.savez("../data/CSDI_tens.npz",
         data=imputed_data,
         countries=countries,
         series=series_list,
         years=year_cols) # 연도 추가

In [ ]:
np.savez("../data/SAITS_tens.npz",
         data=imputed_data_saits,
         countries=countries,
         series=series_list,
         years=year_cols) # 연도 추가

In [ ]:
np.savez("../data/CSDI_tens.npz", data=imputed_csdi, countries=countries, series=series_list)
np.savez("../data/SAITS_tens.npz", data=imputed_saits, countries=countries, series=series_list)

import numpy as np
import pandas as pd

def process_and_save_final(tensor_path, save_path, model_name, df_meta, scalers_dict):
    loaded = np.load(tensor_path, allow_pickle=True)
    data = loaded['data']
    countries = loaded['countries']
    series_list = loaded['series']

    if data.ndim == 4:
        data = np.squeeze(data)

    year_cols = [str(y) for y in range(1990, 2021)]
    final_rows = []

    # [수정] 실제 존재하는 컬럼명 확인 (에러 방지)
    possible_name_cols = ['Series Name', 'Indicator Name', 'series_name']
    name_col = next((c for c in possible_name_cols if c in df_meta.columns), None)

    possible_region_cols = ['Region', 'region']
    region_col = next((c for c in possible_region_cols if c in df_meta.columns), None)

    possible_income_cols = ['Income Group', 'income_group', 'IncomeGroup']
    income_col = next((c for c in possible_income_cols if c in df_meta.columns), None)

    for i, country in enumerate(countries):
        for j, s_code in enumerate(series_list):
            row_data = data[i, :, j]

            # 역스케일링
            if s_code in scalers_dict:
                scaler = scalers_dict[s_code]
                row_data_rescaled = scaler.inverse_transform(row_data.reshape(-1, 1)).flatten()
            else:
                row_data_rescaled = row_data

            # 메타데이터 추출 (안전한 방식)
            meta_info = df_meta[df_meta['Series Code'] == s_code]
            if not meta_info.empty:
                s_name = meta_info[name_col].values[0] if name_col else "Unknown"
            else:
                s_name = "Unknown"

            country_info = df_meta[df_meta['Country Name'] == country]
            if not country_info.empty:
                region = country_info[region_col].values[0] if region_col else "Unknown"
                income = country_info[income_col].values[0] if income_col else "Unknown"
            else:
                region, income = "Unknown", "Unknown"

            new_row = {
                'Country Name': country,
                'Region': region,
                'Income Group': income,
                'Series Code': s_code
            }
            for y_idx, year in enumerate(year_cols):
                new_row[year] = row_data_rescaled[y_idx]

            final_rows.append(new_row)

    df_final = pd.DataFrame(final_rows)
    df_final.to_csv(save_path, index=False, encoding='utf-8-sig')
    print(f"✅ {model_name} 결과 저장 완료: {save_path}")
    return df_final

df_csdi_final = process_and_save_final("../data/CSDI_tens.npz", "../data/WorldBank_CSDI_Final.csv", "CSDI", df_with_meta, scalers)
df_saits_final = process_and_save_final("../data/SAITS_tens.npz", "../data/WorldBank_SAITS_Final.csv", "SAITS", df_with_meta, scalers)

✅ CSDI 결과 저장 완료: ../data/WorldBank_CSDI_Final.csv
✅ SAITS 결과 저장 완료: ../data/WorldBank_SAITS_Final.csv


In [ ]:
df_csdi_final.head(3)

,Country Name,Region,Income Group,Series Code,1990,1991,1992,1993,1994,1995,...,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020
0,Afghanistan,South Asia,Low income,DT.TDS.DPPF.XP.ZS,12.023594,17.079744,11.224402,12.418708,9.723660,16.883823,...,0.313643,0.553390,1.554784,1.875261,2.805854,3.735882,3.548648,2.595223,2.390422,2.244082
1,Afghanistan,South Asia,Low income,EG.ELC.ACCS.ZS,93.932121,81.566582,79.820045,79.851265,78.822029,61.948280,...,43.222019,69.099998,68.290649,89.500000,71.500000,97.699997,97.699997,96.616135,97.699997,97.699997
2,Afghanistan,South Asia,Low income,GFDD.AI.01,7041.013672,10807.021484,-138.157593,3912.552246,6593.897949,7632.990723,...,139.437515,166.112808,155.781158,172.807281,181.087265,182.145248,166.167038,174.299103,183.287323,11428.706055


In [ ]:
df_saits_final.head(3)

,Country Name,Region,Income Group,Series Code,1990,1991,1992,1993,1994,1995,...,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020
0,Afghanistan,South Asia,Low income,DT.TDS.DPPF.XP.ZS,19.835184,20.249981,20.652576,20.053938,19.648247,19.657051,...,0.313643,0.553390,1.554784,1.875261,2.805854,3.735882,3.548648,2.595223,2.390422,2.244082
1,Afghanistan,South Asia,Low income,EG.ELC.ACCS.ZS,16.706064,12.893746,11.316096,12.053926,12.388746,12.578941,...,43.222019,69.099998,68.290649,89.500000,71.500000,97.699997,97.699997,96.616135,97.699997,97.699997
2,Afghanistan,South Asia,Low income,GFDD.AI.01,8159.153809,10465.647461,12165.241211,13111.529297,14836.931641,16945.671875,...,139.437515,166.112808,155.781158,172.807281,181.087265,182.145248,166.167038,174.299103,183.287323,3181.826172


In [ ]:
import numpy as np
from sklearn.metrics import mean_squared_error

# 1. 모델 결과의 차원을 원본(data_3d)과 똑같이 맞춰주는 함수
def reshape_to_original(imputed_data, target_shape):
    # 만약 결과가 dict 형태라면 데이터만 추출
    if isinstance(imputed_data, dict):
        imputed_data = imputed_data["imputation"]

    # 불필요한 차원(크기가 1인 차원)을 제거하고 원본 모양으로 강제 변경
    # .squeeze()는 크기가 1인 차원을 없앱니다.
    cleaned_data = np.squeeze(imputed_data)

    if cleaned_data.shape != target_shape:
        return cleaned_data.reshape(target_shape)
    return cleaned_data

# 2. 결과 정제 (여기서 에러 원인을 잡아줍니다)
clean_csdi = reshape_to_original(imputed_csdi_test, data_3d.shape)
clean_saits = reshape_to_original(imputed_saits_test, data_3d.shape)

# 3. 모델이 채운 값(예측값)들만 추출
# 이제 indices[0], [1], [2]가 clean_csdi의 차원과 완벽히 일치합니다.
pred_csdi = clean_csdi[indices[0][sel_indices], indices[1][sel_indices], indices[2][sel_indices]]
pred_saits = clean_saits[indices[0][sel_indices], indices[1][sel_indices], indices[2][sel_indices]]

# 4. 진짜 RMSE 계산
rmse_csdi_real = np.sqrt(mean_squared_error(true_values, pred_csdi))
rmse_saits_real = np.sqrt(mean_squared_error(true_values, pred_saits))

print(f"🔥 [진짜 대결] 10% 결측치 복구 결과 🔥")
print("-" * 35)
print(f"✅ CSDI  실제 RMSE: {rmse_csdi_real:.6f}")
print(f"✅ SAITS 실제 RMSE: {rmse_saits_real:.6f}")
print("-" * 35)

# 5. 승자 판정
if rmse_csdi_real < rmse_saits_real:
    print(f"🏆 승자: CSDI (오차가 더 적음)")
else:
    print(f"🏆 승자: SAITS (오차가 더 적음)")

🔥 [진짜 대결] 10% 결측치 복구 결과 🔥
-----------------------------------
✅ CSDI  실제 RMSE: 0.617089
✅ SAITS 실제 RMSE: 0.088530
-----------------------------------
🏆 승자: SAITS (오차가 더 적음)
